In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import boto3
import sys
import ipywidgets as widgets

### Utility Functions

In [ ]:
s3 = boto3.client("s3")

# Display Raw
def display(s):
    sys.stdout.write(s)
    sys.stdout.flush()

# List S3 Bucket
def list_bucket(bucket, subfolder):
    resources = []
    is_truncated = True
    continuation_token = None
    while is_truncated:
        if continuation_token: response = s3.list_objects_v2(Bucket=bucket, Prefix=subfolder, ContinuationToken=continuation_token)
        else: response = s3.list_objects_v2(Bucket=bucket, Prefix=subfolder)
        display("#")
        # parse contents
        if 'Contents' in response:
            for obj in response['Contents']:
                resources.append(obj['Key'].split("/")[-1])
        # check if more data is available
        is_truncated = response['IsTruncated']
        continuation_token = response.get('NextContinuationToken')
    display("\n")
    return resources

### Read Input

In [ ]:
bucket = "sliderule-public"
subfolder = "atl24r3/parquet"
granules = list_bucket(bucket, subfolder)
print(f"Read {len(granules)} granules")

In [ ]:
# selector widget for the granule (built from the granules list)
granule_selector = widgets.Dropdown(
    options=[(name, idx) for idx, name in enumerate(granules)],
    value=0,
    description="Granule:",
    layout=widgets.Layout(width="80%"),
)

In [ ]:
# selector widget for the spot (fixed range 1 to 6)
spot_selector = widgets.Dropdown(
    options=list(range(1, 7)),
    value=1,
    description="Spot:",
)

In [ ]:
# cache granule reads so switching spots doesn't re-download the parquet
granule_cache = {}

def load_granule(granule_idx):
    if granule_idx not in granule_cache:
        path = f"s3://{bucket}/{subfolder}/{granules[granule_idx]}"
        granule_cache[granule_idx] = (path, gpd.read_parquet(path))
    return granule_cache[granule_idx]

### Plot Photon Classifications

In [ ]:
# IPython's display (the utility cell defines a local display() that writes raw text)
from IPython.display import display as ipy_display

# class_ph label + color mapping
class_labels = {
    0:  ("unclassified", "#999999"),
    1:  ("other",        "#9467bd"),
    2:  ("ground",       "#2ca02c"),
    40: ("bathymetry",   "#d62728"),
    41: ("sea surface",  "#1f77b4"),
}

# dedicated area the plot is rendered into
class_output = widgets.Output()

def plot_classifications(change=None):
    granule_idx = granule_selector.value
    selected_spot = spot_selector.value

    # remove the current plot and show a loading indicator immediately
    class_output.clear_output(wait=False)
    with class_output:
        print("Loading…")

    # expose selection as globals so the following cells can reuse them
    global granule, gdf, gt, selected_granule
    selected_granule = granule_idx
    granule, gdf = load_granule(granule_idx)

    # select track
    gt = gdf[gdf["spot"] == selected_spot]

    fig, ax = plt.subplots(figsize=(14, 6))

    for cval, (label, color) in class_labels.items():
        sub = gt[gt["class_ph"] == cval]
        if len(sub) == 0:
            continue
        ax.scatter(sub["x_atc"], sub["geoid_corr_h"],
                   s=2, c=color, label=f"{cval}: {label}")

    ax.set_xlabel("x_atc (m)")
    ax.set_ylabel("ortho_h (m)")
    ax.set_title(f"{granule.split('/')[-1]} / spot {selected_spot} — photon classifications")
    ax.legend(markerscale=4, loc="best")
    plt.tight_layout()

    # replace the loading indicator with the finished plot
    class_output.clear_output(wait=True)
    with class_output:
        plt.show()

# redraw automatically whenever a granule or spot is selected
granule_selector.observe(plot_classifications, names="value")
spot_selector.observe(plot_classifications, names="value")

ipy_display(granule_selector, spot_selector, class_output)
plot_classifications()

### Plot Uncertainties

In [ ]:
# IPython's display (the utility cell defines a local display() that writes raw text)
from IPython.display import display as ipy_display

# dedicated area the plot is rendered into
unc_output = widgets.Output()

def plot_uncertainties(change=None):
    granule_idx = granule_selector.value
    selected_spot = spot_selector.value

    # remove the current plot and show a loading indicator immediately
    unc_output.clear_output(wait=False)
    with unc_output:
        print("Loading…")

    granule, gdf = load_granule(granule_idx)

    # select track
    gt = gdf[gdf["spot"] == selected_spot]

    fig, ax = plt.subplots(figsize=(14, 6))

    # sigma_tvu is a positive uncertainty magnitude; scale the colormap to a
    # robust range of the actual data so the variation is visible
    vmin = np.nanpercentile(gt["sigma_tvu"], 1)
    vmax = np.nanpercentile(gt["sigma_tvu"], 99)

    sc = ax.scatter(gt["x_atc"], gt["geoid_corr_h"],
                    s=2, c=gt["sigma_tvu"],
                    cmap="viridis", vmin=vmin, vmax=vmax)

    cbar = fig.colorbar(sc, ax=ax, extend="both")
    cbar.set_label("sigma_tvu (m)")

    ax.set_xlabel("x_atc (m)")
    ax.set_ylabel("ortho_h (m)")
    ax.set_title(f"{granule.split('/')[-1]} / spot {selected_spot} — vertical uncertainties")
    plt.tight_layout()

    # replace the loading indicator with the finished plot
    unc_output.clear_output(wait=True)
    with unc_output:
        plt.show()

# redraw automatically whenever a granule or spot is selected
granule_selector.observe(plot_uncertainties, names="value")
spot_selector.observe(plot_uncertainties, names="value")

ipy_display(granule_selector, spot_selector, unc_output)
plot_uncertainties()